# DriverGuard × UTA-RLDD (Kaggle)

Notebook này chạy pipeline DriverGuard trên UTA-RLDD và xuất **chỉ số liệu** (trace số theo frame, bảng, báo cáo). Không frame / ảnh mặt nào được ghi ra đĩa.

Code phân tích nằm ở `tools/uta_rldd.py` của repo (từ M53); notebook chỉ gọi nó. Kết quả lần chạy 06/10/2026: `eval/results/dataset_uta_rldd.md`.

| Mục | Làm gì |
|---|---|
| 2 | Đọc nhãn 0 / 5 / 10 từ tên file, chia dev / test theo người |
| 3 | Trace khối 2–12 một lần (phần nặng, ~16 giờ trên 4 CPU, cần hai phiên) |
| 4 | Số nền: cảnh báo sai / giờ + CI trên video `0`, tách khởi động, FAR theo cửa sổ, theo từng người, khung DDAW |
| 5 | Bảng tín hiệu nào tách được mức 0 và 10 |
| 6 | Dò ngưỡng và ablation trên fold dev (trên máy nhanh hơn nhiều: `python tools/uta_rldd.py grid`) |
| 7 | Chấm bộ ngưỡng đã chọn trên fold test, một lần |

## Chuẩn bị (một lần)

1. **Dữ liệu:** *Add Input* → dataset `rishab260/uta-reallife-drowsiness-dataset`.
2. **Internet:** *Settings → Internet → On* (cần cho `pip install mediapipe` và tải model).
3. **Code DriverGuard** (repo private), chọn một trong hai:
   - **Cách A, upload zip:** tạo Kaggle Dataset **private** từ file `dist/driverguard-kaggle-code.zip` (code của một commit từ M53 trở đi, có `tools/uta_rldd.py`, + 2 model), rồi *Add Input* vào notebook. Đóng gói lại từ commit khác (chạy ở thư mục gốc repo):

     ```bash
     rm -rf /tmp/dg && mkdir -p /tmp/dg/models && git archive HEAD edge_agent tools config | tar -x -C /tmp/dg \
       && cp models/face_landmarker.task models/open_closed_eye.onnx /tmp/dg/models/ && git rev-parse HEAD > /tmp/dg/BUNDLE_COMMIT \
       && rm -f dist/driverguard-kaggle-code.zip && (cd /tmp/dg && zip -q -r "$OLDPWD/dist/driverguard-kaggle-code.zip" .)
     ```
   - **Cách B, clone bằng token:** *Add-ons → Secrets* → thêm secret tên `GITHUB_TOKEN` (token chỉ cần quyền đọc repo) và bật cho notebook này.
4. **Accelerator:** None (CPU). GPU không giúp gì ở đây.

## Cách chạy

1. **Chạy thử:** để `RUN_MODE = "smoke"`, bấm *Run All*. Chỉ chạy 1 người (3 video nhỏ nhất), vài phút. Ô cuối phải in `FINISHED`. Ô 3 in ước lượng thời gian cho lần chạy đầy đủ.
2. **Chạy đầy đủ:** đổi `RUN_MODE = "full"`, bấm **Save Version → Save & Run All (Commit)**. Phiên chạy nền, tối đa 12 giờ, không cần mở trình duyệt.
3. **Nếu chưa trace hết trong một phiên:** notebook tự dừng nhận video mới sau `TRACE_BUDGET_H` giờ và vẫn lưu output. Chạy lại lần nữa với output của phiên trước gắn làm input (*Add Input → Notebook Output* của chính notebook này): video đã trace được bỏ qua.

**Trace gắn với phiên bản code:** các cột trong trace là của commit đã dùng để trace (ghi trong `run_info.json`). Code đổi khối 2–6 hoặc thêm cột trace thì phải trace lại (nhiều giờ); đổi ngưỡng / logic khối 7–12 thì chỉ cần replay.

## Gửi lại gì

- Toàn bộ thư mục **Output** của phiên chạy đầy đủ (nút *Download All*), gồm `traces/`.
- **Log** của phiên (tab *Logs*).


## 0. Tham số

In [ ]:
import os
import time

T_START = time.time()

RUN_MODE = "smoke"          # "smoke": one subject, a few minutes | "full": every fold in FOLDS

DATA_ROOT = None            # None = find the UTA-RLDD folder below /kaggle/input
REPO_URL = "github.com/AI20K-Build-Phase-Cohort-4/P-135.git"
REPO_REF = "main"           # branch / tag cloned when no code bundle is attached

FOLDS = [1, 2, 3, 4]        # folds traced in this session (the Kaggle copy has no fold 5)
DEV_FOLDS = [1, 2]          # thresholds are searched on these subjects only
TEST_FOLDS = [3, 4]         # held out: ~4 h of awake video, 0 false alarms there gives a 95 % upper bound < 1.0 / h
SCORE_TEST = False          # True only once: baseline numbers on the held-out folds
FINAL_SETS = []             # chosen thresholds, e.g. ["risk.drowsy.l1=45"]: scored once on TEST_FOLDS (section 7)

WORKERS = os.cpu_count() or 4
TRACE_BUDGET_H = 9.0        # no new video starts after this many hours from the start of the notebook
TOTAL_BUDGET_H = 11.3       # the threshold search stops here (a Kaggle CPU session ends at 12 h)
RUN_GRID = True
GRID = None                 # None = uta_rldd.default_grid(); or [("group", ["a.b=1", "c.d=2"]), ...]

MEDIAPIPE_SPEC = "mediapipe>=0.10.20"   # as requirements.txt

## 1. Cài đặt: code DriverGuard, thư viện, model

In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("UTA_WORK", "uta_out")).resolve()
WORK.mkdir(parents=True, exist_ok=True)
REPO = Path(os.environ.get("DRIVERGUARD_REPO", "/tmp/P-135"))   # outside /kaggle/working: the code is not an output
os.chdir(WORK)


def find_below(root, match, max_depth=5):
    """Directories / files below root for which match(path) is true, not deeper than max_depth."""
    out, stack = [], [(Path(root), 0)]
    while stack:
        d, depth = stack.pop()
        try:
            entries = sorted(d.iterdir())
        except OSError:
            continue
        for p in entries:
            if match(p):
                out.append(p)
            elif p.is_dir() and depth + 1 < max_depth:
                stack.append((p, depth + 1))
    return sorted(out)


# ---- 1. code: a bundle attached as a dataset (folder or zip), else a clone with the GITHUB_TOKEN secret
if not (REPO / "edge_agent" / "config.py").exists():
    bundles = find_below(INPUT, lambda p: p.is_dir() and (p / "edge_agent" / "config.py").exists()) if ON_KAGGLE else []
    zips = find_below(INPUT, lambda p: p.suffix == ".zip" and "driverguard" in p.name.lower()) if ON_KAGGLE else []
    if bundles:
        shutil.copytree(bundles[0], REPO, dirs_exist_ok=True)
        subprocess.run(["chmod", "-R", "u+w", str(REPO)], check=False)   # /kaggle/input is read-only
        print(f"[setup] code copied from {bundles[0]}")
    elif zips:
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(REPO)
        print(f"[setup] code unpacked from {zips[0]}")
    else:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception as e:
            raise SystemExit(
                "No DriverGuard code found. Attach the code bundle (dist/driverguard-kaggle-code.zip uploaded as a "
                f"private dataset) or add the GITHUB_TOKEN secret. ({type(e).__name__})"
            )
        r = subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_REF, f"https://x-access-token:{token}@{REPO_URL}", str(REPO)],
            capture_output=True, text=True,
        )
        if r.returncode:
            raise SystemExit("git clone failed: " + r.stderr.replace(token, "***")[-800:])
        subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", f"https://{REPO_URL}"], check=False)
        print(f"[setup] cloned {REPO_URL} @ {REPO_REF}")
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
if not commit and (REPO / "BUNDLE_COMMIT").exists():   # written into the code bundle when it was made
    commit = (REPO / "BUNDLE_COMMIT").read_text().strip()
for pkg in ("edge_agent", "tools"):   # a dataset upload may drop empty files; packages still need to import
    assert (REPO / pkg).is_dir(), f"{pkg}/ missing in {REPO}"
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(WORK))
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO), str(WORK), os.environ.get("PYTHONPATH", "")])
os.environ.setdefault("OMP_NUM_THREADS", "1")   # one process per core: no thread oversubscription

# ---- 2. packages (checked in a child process: the kernel itself never imports MediaPipe)
CHECK = "import mediapipe, cv2, numpy, onnxruntime, yaml; print(mediapipe.__version__, cv2.__version__, numpy.__version__, onnxruntime.__version__)"
probe = subprocess.run([sys.executable, "-c", CHECK], capture_output=True, text=True)
if probe.returncode:
    print("[setup] installing packages ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", MEDIAPIPE_SPEC, "onnxruntime>=1.18.0", "pyyaml>=6.0"], check=True)
    probe = subprocess.run([sys.executable, "-c", CHECK], capture_output=True, text=True)
    if probe.returncode:
        raise SystemExit("packages still not importable:\n" + probe.stderr[-1500:])
versions = dict(zip(("mediapipe", "opencv", "numpy", "onnxruntime"), probe.stdout.split()))
print(f"[setup] python {sys.version.split()[0]}, {versions}")
assert sys.version_info >= (3, 11), "DriverGuard needs Python >= 3.11"

# ---- 3. models (block 3 landmarks, block 5 eye CNN; the phone model is not used here)
from tools.download_models import MODELS  # noqa: E402

LOCAL_SHA256 = {   # the files the project's other eval results were made with
    "face_landmarker.task": "64184e229b263107bc2b804c6625db1341ff2bb731874b0bcc2fe6544e0bc9ff",
    "open_closed_eye.onnx": "4daa100034482525a26c9afb9297c16580a531189e66e3d2b2ac7d32becfd593",
}
(REPO / "models").mkdir(exist_ok=True)
model_sha = {}
for name, (url, sha384) in MODELS.items():
    dest = REPO / "models" / name
    if not dest.exists():
        print(f"[setup] downloading {name}")
        urllib.request.urlretrieve(url, dest)
        if sha384 and hashlib.sha384(dest.read_bytes()).hexdigest() != sha384:
            dest.unlink()
            raise SystemExit(f"{name}: checksum mismatch")
    model_sha[name] = hashlib.sha256(dest.read_bytes()).hexdigest()
    same = model_sha[name] == LOCAL_SHA256.get(name)
    print(f"[setup] {name} sha256 {model_sha[name][:16]} " + ("= local file" if same else "DIFFERS from the local file (noted in run_info.json)"))

from edge_agent.config import load_thresholds  # noqa: E402

TH = load_thresholds()
shutil.copy(REPO / "config" / "thresholds.yaml", WORK / "thresholds_used.yaml")
RUN_INFO = {
    "mode": RUN_MODE, "commit": commit or "bundle (no git)", "python": sys.version.split()[0], "versions": versions,
    "models_sha256": model_sha, "workers": WORKERS, "cpu_count": os.cpu_count(),
    "folds": FOLDS, "dev_folds": DEV_FOLDS, "test_folds": TEST_FOLDS,
}
print(f"[setup] repo {REPO} commit {RUN_INFO['commit']}, output {WORK}, {WORKERS} workers")

## 2. Danh sách video, chia dev / test theo người

In [ ]:
from tools import uta_rldd as U  # noqa: N812

if DATA_ROOT is None:
    if "UTA_DATA_ROOT" in os.environ:
        DATA_ROOT = os.environ["UTA_DATA_ROOT"]
    else:
        folds = find_below(INPUT, lambda p: p.is_dir() and p.name.lower().startswith("fold"), max_depth=5)
        assert folds, "UTA-RLDD not found below /kaggle/input: add the dataset rishab260/uta-reallife-drowsiness-dataset"
        DATA_ROOT = str(folds[0].parent)
ALL_VIDEOS, skipped = U.discover(DATA_ROOT)
print(f"[index] {DATA_ROOT}: {len(ALL_VIDEOS)} labelled videos, {sum(v.size_mb for v in ALL_VIDEOS) / 1000:.1f} GB")
if skipped:
    print(f"[index] {len(skipped)} video files with a name that is not a label, left out: {skipped}")
assert ALL_VIDEOS, "no video found"

index = []
for v in ALL_VIDEOS:
    index.append({**U.video_rows([v])[0], **U.probe(v.path)})
for row in index:
    row["path"] = os.path.relpath(row["path"], DATA_ROOT)
U.write_csv(WORK / "uta_index.csv", index)
total_frames = sum(r["frames"] for r in index)
total_min = sum(r["minutes"] for r in index if r["minutes"] == r["minutes"])
print(f"[index] {total_min / 60:.1f} h of video, {total_frames} frames")
by_fold = {}
for r in index:
    f = by_fold.setdefault(r["fold"], {"subjects": set(), 0: 0, 5: 0, 10: 0, "min": 0.0, "gb": 0.0})
    f["subjects"].add(r["subject"])
    f[r["label"]] += 1
    f["min"] += r["minutes"]
    f["gb"] += r["size_mb"] / 1000
for fold, f in sorted(by_fold.items()):
    print(f"[index] fold {fold}: {len(f['subjects'])} subjects, videos 0/5/10 = {f[0]}/{f[5]}/{f[10]}, {f['min'] / 60:.1f} h, {f['gb']:.1f} GB")
odd = [r for r in index if not (r["fps"] and 5 <= r["fps"] <= 61 and r["frames"] > 0)]
if odd:
    print("[index] unusual FPS / frame count (time in the trace = frame index / FPS):", [(r["key"], r["fps"], r["frames"]) for r in odd])
res = sorted({(r["width"], r["height"]) for r in index})
print(f"[index] resolutions: {res}")

if RUN_MODE == "smoke":   # the subject with all three levels and the smallest files
    size = {}
    for v in ALL_VIDEOS:
        s = size.setdefault((v.fold, v.subject), {"mb": 0.0, "labels": set()})
        s["mb"] += v.size_mb
        s["labels"].add(v.label)
    fold, subject = min((s["mb"], k) for k, s in size.items() if s["labels"] == {0, 5, 10})[1]
    VIDEOS = [v for v in ALL_VIDEOS if (v.fold, v.subject) == (fold, subject)]
    DEV_FOLDS, TEST_FOLDS = [fold], []
    RUN_INFO |= {"dev_folds": DEV_FOLDS, "test_folds": TEST_FOLDS, "smoke_subject": subject}
    print(f"[index] SMOKE TEST: subject {subject} of fold {fold} only, {sum(v.size_mb for v in VIDEOS):.0f} MB")
else:
    VIDEOS = [v for v in ALL_VIDEOS if v.fold in FOLDS]
    assert not set(DEV_FOLDS) & set(TEST_FOLDS), "a fold cannot be in both splits"

# the split goes into the repo as eval/splits/uta_rldd_{dev,test}.txt: one subject per line
(WORK / "splits").mkdir(exist_ok=True)
for name, folds in (("dev", DEV_FOLDS), ("test", TEST_FOLDS)):
    subjects = sorted({f"F{v.fold}_S{v.subject}" for v in ALL_VIDEOS if v.fold in folds})
    (WORK / "splits" / f"uta_rldd_{name}.txt").write_text("\n".join(subjects) + "\n")
    print(f"[index] split {name}: folds {folds}, {len(subjects)} subjects")

## 3. Trace: chạy khối 2–12 một lần trên mỗi video

Phần nặng nhất. Mỗi video một file `traces/<key>.pkl.gz` (chỉ số theo frame). Model điện thoại tắt: người trong UTA-RLDD không lái xe, và replay cũng không có cờ điện thoại nên hai bên khớp nhau.

In [ ]:
TRACES = WORK / "traces"
resume = [p for p in find_below(INPUT, lambda p: p.is_dir() and p.name == "traces", max_depth=4)] if ON_KAGGLE else []
if resume:
    print(f"[trace] earlier traces found in {[str(p) for p in resume]}")
t0 = time.time()
trace_log = U.run_traces(
    VIDEOS, TRACES, TH, resume_dirs=resume, workers=WORKERS, deadline=T_START + TRACE_BUDGET_H * 3600
)
if trace_log:
    log_path = WORK / "trace_log.csv"
    U.write_csv(log_path, trace_log)
    frames = sum(r["frames"] for r in trace_log)
    wall = time.time() - t0
    rate = frames / wall
    failed = [r["key"] for r in trace_log if r["error"]]
    print(f"[trace] {frames} frames in {wall / 60:.1f} min = {rate:.0f} frames/s with {WORKERS} workers; failed: {failed or 'none'}")
    if rate > 0:
        print(
            f"[trace] estimate for all {len(ALL_VIDEOS)} videos ({total_frames} frames): {total_frames / rate / 3600:.1f} h "
            f"at this speed (budget TRACE_BUDGET_H = {TRACE_BUDGET_H} h). High-resolution videos decode slower, "
            "so a smoke run on the smallest files underestimates it; what is left over goes to a second session"
        )
    low = [(r["key"], r["face"]) for r in trace_log if not r["error"] and r["face"] < 0.8]
    if low:
        print(f"[trace] face found on < 80 % of frames (rotated video? face out of view?): {low}")
FOUND = U.find_traces([TRACES])
missing = [v.key for v in VIDEOS if v.key not in FOUND]
RUN_INFO |= {"videos_wanted": len(VIDEOS), "videos_traced": len(VIDEOS) - len(missing), "videos_missing": missing}
print(f"[trace] traced {len(VIDEOS) - len(missing)}/{len(VIDEOS)} videos; missing: {missing or 'none'}")
print(f"[trace] traces on disk: {sum(p.stat().st_size for p in FOUND.values()) / 1e6:.0f} MB")

## 4. Số nền với ngưỡng hiện tại

Replay khối 7–12 trên trace. Mặc định chỉ chấm các fold dev; fold test chỉ chấm khi `SCORE_TEST = True`.

In [ ]:
def items_of(folds):
    return [(v, FOUND[v.key]) for v in VIDEOS if v.fold in folds and v.key in FOUND]


POOL = U.replay_pool(WORKERS)
SPLITS = [("dev", DEV_FOLDS)] + ([("test", TEST_FOLDS)] if SCORE_TEST and TEST_FOLDS else [])
ROWS = {}
for split, folds in SPLITS:
    items = items_of(folds)
    if not items:
        print(f"[baseline] {split}: no trace for folds {folds}, skipped")
        continue
    t0 = time.time()
    rows = ROWS[split] = U.replay_rows(items, TH, POOL, extras=True)
    same = sum(r["a_all"] == r["a_traced"] for r in rows)
    print(
        f"[baseline] {split}: {len(rows)} videos replayed in {time.time() - t0:.0f} s; replay gives the same number of "
        f"alerts as the traced run on {same}/{len(rows)} videos"
    )
    U.write_csv(WORK / f"uta_videos_{split}.csv", rows)
    note = f"Split: {split} (folds {folds}). Thresholds: config/thresholds.yaml at {RUN_INFO['commit'][:12]}. Mode: {RUN_MODE}."
    text = U.baseline_report(rows, TH, f"DriverGuard - UTA-RLDD, current thresholds ({split})", note)
    (WORK / f"uta_baseline_{split}.md").write_text(text, encoding="utf-8")
    print(text)

## 5. Tín hiệu nào tách được mức 0 và 10

In [ ]:
for split, rows in ROWS.items():
    text = U.signals_report(rows, f"DriverGuard - UTA-RLDD, which signals follow the drowsiness level ({split})")
    (WORK / f"uta_signals_{split}.md").write_text(text, encoding="utf-8")
    print(text)

## 6. Dò ngưỡng và ablation (chỉ trên fold dev)

Mỗi cấu hình đổi 1–2 khoá. Không dò ngưỡng microsleep / L3: UTA-RLDD không cho biết microsleep nằm ở đâu.

In [ ]:
GRID_RESULTS = []
if RUN_GRID and items_of(DEV_FOLDS):
    configs = GRID or U.default_grid()
    if RUN_MODE == "smoke":
        configs = configs[:3] + configs[-2:]   # only checks that the search runs
    print(f"[grid] {len(configs)} configurations on the dev folds {DEV_FOLDS}")
    GRID_RESULTS = U.run_configs(
        configs, items_of(DEV_FOLDS), TH, POOL, WORK / "uta_grid_dev.csv", deadline=T_START + TOTAL_BUDGET_H * 3600
    )
    note = f"Split: dev (folds {DEV_FOLDS}), one or two keys changed at a time from config/thresholds.yaml at {RUN_INFO['commit'][:12]}."
    text = U.grid_report(GRID_RESULTS, "DriverGuard - UTA-RLDD, threshold search and ablation (dev)", note)
    (WORK / "uta_grid_dev.md").write_text(text, encoding="utf-8")
    print(text)

## 7. Chấm bộ ngưỡng đã chọn trên fold test (chỉ khi `FINAL_SETS` có giá trị)

Chạy **một lần** sau khi đã chốt bộ ngưỡng từ bảng dev.

In [ ]:
if FINAL_SETS and items_of(TEST_FOLDS):
    items = items_of(TEST_FOLDS)
    final = U.run_configs([("current", []), ("chosen", list(FINAL_SETS))], items, TH, POOL, WORK / "uta_final_test.csv")
    text = U.grid_report(final, "DriverGuard - UTA-RLDD, chosen thresholds on the held-out folds", f"Split: test (folds {TEST_FOLDS}).")
    th_new = U.make_thresholds(TH, list(FINAL_SETS))
    rows = U.replay_rows(items, th_new, POOL, extras=True)
    U.write_csv(WORK / "uta_videos_test_chosen.csv", rows)
    text += "\n" + U.baseline_report(
        rows, th_new, "DriverGuard - UTA-RLDD, chosen thresholds (test)", "Thresholds changed: " + ", ".join(FINAL_SETS) + "."
    )
    (WORK / "uta_final_test.md").write_text(text, encoding="utf-8")
    print(text)
else:
    print("[final] FINAL_SETS is empty: the held-out folds are not scored with new thresholds in this run")

## 8. Kết thúc

In [ ]:
if POOL is not None:
    POOL.shutdown()
RUN_INFO |= {"hours": round((time.time() - T_START) / 3600, 2), "grid_configs": len(GRID_RESULTS)}
(WORK / "run_info.json").write_text(json.dumps(RUN_INFO, indent=2))
print(json.dumps(RUN_INFO, indent=2))
print("\nOutput files:")
for p in sorted(WORK.iterdir()):
    if p.is_dir():
        files = [f for f in p.rglob("*") if f.is_file()]
        print(f"  {p.name}/  {len(files)} files, {sum(f.stat().st_size for f in files) / 1e6:.1f} MB")
    else:
        print(f"  {p.name}  {p.stat().st_size / 1e3:.1f} kB")
left = RUN_INFO["videos_missing"]
print(f"\n=== UTA-RLDD NOTEBOOK FINISHED (mode={RUN_MODE}, {RUN_INFO['hours']} h) ===")
print("all videos traced" if not left else f"{len(left)} videos NOT traced yet: run again with this output attached as input")